# 1장 1강: 가설검정 프레임워크의 이해 — 실습문제

## 실습 목표

- 분석 상황에 맞게 귀무가설(H₀)과 대립가설(H₁)을 설정할 수 있다.
- `scipy.stats.ttest_1samp()`로 단일표본 t검정을 수행할 수 있다.
- 유의수준과 p-value를 비교하여 귀무가설 기각 여부를 판단할 수 있다.
- 제1종 오류와 제2종 오류를 분석 상황에 맞게 설명할 수 있다.
- 연구 목적에 따라 양측검정과 단측검정을 구분할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- scipy.stats
- `ames_housing.csv`

이번 실습에서는 미국 아이오와주 Ames 지역의 주택 자료를 사용합니다.

주요 컬럼은 다음과 같습니다.

| 컬럼 | 의미 |
|---|---|
| `SalePrice` | 주택 판매가격 |
| `GrLivArea` | 지상층 생활면적 |
| `OverallQual` | 주택의 전반적인 품질 점수(1~10) |
| `YearBuilt` | 건축연도 |

> 모든 검정의 유의수준은 별도 지시가 없으면 `α = 0.05`로 설정합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. Ames Housing 데이터를 `df`에 불러오세요.
3. 데이터의 행과 열 개수, 컬럼명, 상위 5개 행을 확인하세요.

In [1]:
# 실습 준비 코드를 작성하세요.

import os
from pathlib import Path
import pandas as pd
from scipy import stats

# 1. 파일 경로 설정 및 데이터 불러오기 (동일 폴더 또는 data/ 폴더 순차 탐색)
ROOT = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
csv_path = ROOT / "ames_housing.csv"
if not csv_path.exists():
    csv_path = ROOT / "data" / "ames_housing.csv"

# Check file existence
files = os.listdir(csv_path.parent)
print("Files in dir:", [f for f in files if 'csv' in f or 'ames' in f])

# 상대 경로로 안전하게 로드
df = pd.read_csv(csv_path)

# 2. 데이터 크기, 컬럼명, 상위 5개 행 확인
print("=" * 50)
print(f"데이터 크기 (행, 열): {df.shape}")
print("-" * 50)
print("컬럼 목록:")
print(df.columns.tolist()[:10], "... (총", len(df.columns), "개 컬럼)")
print("-" * 50)
print("상위 5개 행 (주요 실습 컬럼):")
target_cols = ["SalePrice", "GrLivArea", "OverallQual", "YearBuilt"]
available_cols = [col for col in target_cols if col in df.columns]
print(df[available_cols].head())
print("=" * 50)

Files in dir: ['ames_housing 컬럼정의서.xlsx', 'ames_housing.csv', 'cookie_cats.csv']
데이터 크기 (행, 열): (1460, 10)
--------------------------------------------------
컬럼 목록:
['SalePrice', 'GrLivArea', 'LotArea', 'OverallQual', 'KitchenQual', 'CentralAir', 'HeatingQC', 'PavedDrive', 'Neighborhood', 'YearBuilt'] ... (총 10 개 컬럼)
--------------------------------------------------
상위 5개 행 (주요 실습 컬럼):
   SalePrice  GrLivArea  OverallQual  YearBuilt
0     208500       1710            7       2003
1     181500       1262            6       1976
2     223500       1786            7       2001
3     140000       1717            7       1915
4     250000       2198            8       2000


---

## 필수 1. 평균 판매가격에 대한 양측검정

### 문제 1-1. 평균 판매가격이 180,000달러와 다른가?

#### 문제 설명

Ames 지역의 주택 평균 판매가격에 대한 기존 기준이 **180,000달러**라고 가정합니다.  
제공된 표본의 평균 판매가격이 이 기준과 다른지 단일표본 t검정으로 확인하세요.

#### 요구사항

1. `SalePrice`에서 결측치를 제거하여 `sale_price`에 저장하세요.
2. 표본 수와 표본평균을 출력하세요.
3. 다음 가설을 코드 주석이나 마크다운으로 작성하세요.
   - H₀: Ames 지역의 평균 판매가격은 180,000달러이다.
   - H₁: Ames 지역의 평균 판매가격은 180,000달러가 아니다.
4. 유의수준 `alpha`를 0.05로 설정하세요.
5. `stats.ttest_1samp()`와 `alternative="two-sided"`를 사용해 양측검정을 수행하세요.
6. t통계량과 p-value를 소수점 넷째 자리까지 출력하세요.
7. p-value와 유의수준을 비교해 귀무가설 기각 여부를 출력하세요.

#### 해석 질문

**Q1.** 이 문제에서 양측검정을 사용하는 이유는 무엇인가요?  
**Q2.** 검정 결과에 따라 귀무가설을 기각할 수 있나요?  
**Q3.** 귀무가설을 기각하지 못했다면 평균 판매가격이 정확히 180,000달러라고 증명된 것인가요?

#### 제출 결과

- 가설 설정
- 표본 수와 표본평균
- 단일표본 t검정 코드
- t통계량과 p-value
- 귀무가설 판단
- Q1~Q3 답변

In [2]:
# [가설 설정]
# H₀ (귀무가설): Ames 지역 주택의 모평균 판매가격은 180,000달러이다. (μ = 180,000)
# H₁ (대립가설): Ames 지역 주택의 모평균 판매가격은 180,000달러가 아니다. (μ ≠ 180,000)

# 1. 결측치 제거
sale_price = df["SalePrice"].dropna()

# 2. 표본 수와 표본평균 출력
n = len(sale_price)
sample_mean = sale_price.mean()
print(f"[필수 1] 표본 수(n): {n:,}개")
print(f"[필수 1] 표본 평균: ${sample_mean:,.2f}")

# 3. 단일표본 t-검정 수행 (양측검정)
popmean_val = 180000
alpha = 0.05
t_stat, p_value = stats.ttest_1samp(
    sale_price, popmean=popmean_val, alternative="two-sided"
)

# 4. 결과 출력
print(f"t-통계량: {t_stat:.4f}")
print(f"p-value : {p_value:.4f}")

# 5. 귀무가설 기각 여부 판단
if p_value < alpha:
    print(
        f"결과: p-value({p_value:.4f}) < {alpha} 이므로 귀무가설(H₀)을 기각합니다."
    )
    print("해석: 평균 판매가격이 180,000달러와 다르다는 통계적 근거가 있습니다.")
else:
    print(
        f"결과: p-value({p_value:.4f}) >= {alpha} 이므로 귀무가설(H₀)을 기각할 수 없습니다."
    )
    print(
        "해석: 평균 판매가격이 180,000달러와 다르다고 볼 충분한 통계적 근거가 부족합니다."
    )

[필수 1] 표본 수(n): 1,460개
[필수 1] 표본 평균: $180,921.20
t-통계량: 0.4431
p-value : 0.6578
결과: p-value(0.6578) >= 0.05 이므로 귀무가설(H₀)을 기각할 수 없습니다.
해석: 평균 판매가격이 180,000달러와 다르다고 볼 충분한 통계적 근거가 부족합니다.


### 2. 필수 1 해석 질문 답변 (Q1 ~ Q3)

* **Q1. 이 문제에서 양측검정을 사용하는 이유는 무엇인가요?**  
  연구 질문이 "180,000달러보다 높은가?" 혹은 "낮은가?"처럼 사전에 특정 방향을 가리키지 않고, **단순히 "기존 기준값인 180,000달러와 차이가 있는가(달라졌는가, $\mu \neq 180,000$)?"**를 검증하기 때문입니다. 기준값보다 큰 경우와 작은 경우를 모두 유의한 차이로 고려해야 하므로 양측검정을 사용합니다.
* **Q2. 검정 결과에 따라 귀무가설을 기각할 수 있나요?**  
  계산된 `p-value`가 사전에 정한 유의수준 `alpha = 0.05`보다 작다면 귀무가설을 기각할 수 있고, 크거나 같다면 기각할 수 없습니다. (Ames Housing 원본 데이터의 경우 표본평균이 180,921달러 수준으로 기준값 180,000달러와 차이가 미미하여 $p > 0.05$로 기각 실패가 나오거나, 표본 크기가 매우 커서 미세한 차이라도 $p < 0.05$로 기각될 수 있으며, 출력된 p-value의 대소 비교에 따라 결론을 내립니다.)
* **Q3. 귀무가설을 기각하지 못했다면 평균 판매가격이 정확히 180,000달러라고 증명된 것인가요?**  
  **절대 아닙니다.** 귀무가설을 기각하지 못했다는 것은 "현재 가진 표본 데이터의 근거만으로는 180,000달러와 다르다고 단정하기에 증거가 불충분하다(증거의 부재, Absence of Evidence)"는 뜻일 뿐, "평균이 정확히 180,000달러임이 참으로 입증되었다(부재의 증거, Evidence of Absence)"는 뜻이 아닙니다.

---

## 필수 2. 양측검정과 단측검정 비교

### 문제 2-1. 평균 품질 점수는 6점과 다른가, 그리고 6점보다 높은가?

#### 문제 설명

주택의 전반적인 품질을 나타내는 `OverallQual`의 기준 점수를 **6점**으로 설정했습니다.  
같은 표본으로 다음 두 연구 질문을 각각 검정하고 결과를 비교하세요.

- 질문 A: 평균 품질 점수가 6점과 **다른가?**
- 질문 B: 평균 품질 점수가 6점보다 **높은가?**

#### 요구사항

1. `OverallQual`에서 결측치를 제거하여 `overall_qual`에 저장하세요.
2. 표본 수와 표본평균을 출력하세요.
3. 질문 A의 가설을 작성하고 `alternative="two-sided"`로 양측검정을 수행하세요.
4. 질문 B의 가설을 작성하고 `alternative="greater"`로 단측검정을 수행하세요.
5. 두 검정의 t통계량과 p-value를 출력하세요.
6. 유의수준 0.05를 기준으로 각 검정의 귀무가설 기각 여부를 출력하세요.
7. 같은 데이터인데 두 p-value가 다르게 계산되는 이유를 설명하세요.

#### 해석 질문

**Q1.** 질문 A와 질문 B의 대립가설은 어떻게 다른가요?  
**Q2.** 양측검정과 단측검정의 결론은 각각 무엇인가요?  
**Q3.** 분석 결과를 먼저 확인한 뒤 p-value를 작게 만들기 위해 단측검정으로 바꿔도 되나요?  
**Q4.** 실제 평균 품질 점수가 6점인데도 검정 결과가 유의하다고 판단했다면 어떤 오류인가요?

#### 제출 결과

- 두 검정의 가설
- 표본 수와 표본평균
- 양측검정 및 단측검정 코드
- 두 검정의 t통계량과 p-value
- 검정별 귀무가설 판단
- Q1~Q4 답변

In [3]:
# [가설 설정]
# 질문 A (양측검정)
# H₀: 평균 품질 점수는 6점이다. (μ = 6)
# H₁: 평균 품질 점수는 6점이 아니다. (μ ≠ 6)

# 질문 B (우측 단측검정)
# H₀: 평균 품질 점수는 6점 이하이다. (μ ≤ 6)
# H₁: 평균 품질 점수는 6점보다 높다. (μ > 6)

# 1. 결측치 제거
overall_qual = df["OverallQual"].dropna()
n_qual = len(overall_qual)
mean_qual = overall_qual.mean()

print(f"[필수 2] 표본 수: {n_qual:,}개 | 표본 평균: {mean_qual:.4f}점")

# 2. 질문 A: 양측검정 (two-sided)
t_stat_a, p_val_a = stats.ttest_1samp(
    overall_qual, popmean=6, alternative="two-sided"
)

# 3. 질문 B: 단측검정 (greater)
t_stat_b, p_val_b = stats.ttest_1samp(
    overall_qual, popmean=6, alternative="greater"
)

print(f"질문 A (양측검정) - t통계량: {t_stat_a:.4f}, p-value: {p_val_a:.4f}")
print(f"질문 B (단측검정) - t통계량: {t_stat_b:.4f}, p-value: {p_val_b:.4f}")

# 4. 판정
alpha = 0.05
print(
    f"질문 A 판정: {'귀무가설 기각 (차이 있음)' if p_val_a < alpha else '기각 불가 (차이 근거 부족)'}"
)
print(
    f"질문 B 판정: {'귀무가설 기각 (6점보다 유의하게 높음)' if p_val_b < alpha else '기각 불가'}"
)

# 5. 두 p-value가 다른 이유
print("-" * 50)
print(f"비교: 양측 p-value ({p_val_a:.4f}) vs 단측 p-value ({p_val_b:.4f})")
print(
    f"표본평균({mean_qual:.2f})이 6보다 크므로, 단측검정 p-value는 양측검정의 정확히 절반(1/2)입니다."
)

[필수 2] 표본 수: 1,460개 | 표본 평균: 6.0993점
질문 A (양측검정) - t통계량: 2.7439, p-value: 0.0061
질문 B (단측검정) - t통계량: 2.7439, p-value: 0.0031
질문 A 판정: 귀무가설 기각 (차이 있음)
질문 B 판정: 귀무가설 기각 (6점보다 유의하게 높음)
--------------------------------------------------
비교: 양측 p-value (0.0061) vs 단측 p-value (0.0031)
표본평균(6.10)이 6보다 크므로, 단측검정 p-value는 양측검정의 정확히 절반(1/2)입니다.


### 2. 필수 2 해석 질문 답변 (Q1 ~ Q4)

* **Q1. 질문 A와 질문 B의 대립가설은 어떻게 다른가요?**  
  질문 A의 대립가설은 방향이 없는 단순 차이($\mu \neq 6$)인 반면, 질문 B의 대립가설은 **사전에 특정 방향(점수 상승, $\mu > 6$)을 지정**하여 검정합니다.
* **Q2. 양측검정과 단측검정의 결론은 각각 무엇인가요?**  
  Ames 주택의 평균 품질 점수는 약 6.09~6.10점으로 6점보다 높습니다. 따라서 양측검정에서는 "평균 점수가 6점과 다르다"는 결론을 내리고, 단측검정에서는 "평균 점수가 6점보다 통계적으로 유의하게 높다"는 결론을 내립니다.
* **Q3. 분석 결과를 먼저 확인한 뒤 p-value를 작게 만들기 위해 단측검정으로 바꿔도 되나요?**  
  **절대 안 됩니다.** 이는 전형적인 **데이터 스누핑(Data Snooping) 및 P-해킹(P-hacking)**입니다. 단측검정은 데이터를 수집하기 전 기획 단계에서 확립된 사전 가설(A priori)이 있을 때만 써야 합니다. 결과를 보고 p-value를 억지로 절반으로 깎아 유의하게 만드는 것은 제1종 오류(헛경보) 확률을 의도적으로 2배 올리는 심각한 통계 왜곡입니다.
* **Q4. 실제 평균 품질 점수가 6점인데도 검정 결과가 유의하다고 판단했다면 어떤 오류인가요?**  
  **제1종 오류 (Type I Error, $\alpha$)**입니다. 실제로는 차이나 효과가 없는데(귀무가설 참), 우연한 표본 오차에 속아 "유의한 차이가 있다"고 잘못 판단(False Positive)한 것입니다.

---

## 과제 1. 평가 문항 기반 독립 과제

### 문제 3-1. 평균 지상층 생활면적이 1,550보다 작은가?

#### 문제 설명

한 부동산 분석가는 Ames 지역 주택의 평균 지상층 생활면적이 **1,550보다 작다**고 주장합니다.  
`GrLivArea`를 사용하여 이 주장을 단일표본 t검정으로 확인하세요.

> 이 문제는 필수 문제에서 연습한 가설 설정과 단측검정 절차를 새로운 변수에 적용하는 문제입니다.

#### 요구사항

1. `GrLivArea`에서 결측치를 제거하여 분석 표본을 준비하세요.
2. 표본 수와 표본평균을 확인하세요.
3. 연구 주장에 맞는 귀무가설과 대립가설을 작성하세요.
4. 양측검정과 단측검정 중 적절한 검정 방향을 선택하고 그 이유를 설명하세요.
5. 단일표본 t검정을 수행하세요.
6. t통계량과 p-value를 소수점 넷째 자리까지 출력하세요.
7. 유의수준 0.05를 기준으로 귀무가설 기각 여부를 판단하세요.
8. 분석가의 주장을 뒷받침할 통계적 근거가 있는지 해석하세요.

#### 해석 질문

**Q1.** 이 문제에서 적절한 `alternative` 값은 무엇인가요?  
**Q2.** p-value는 귀무가설이 참일 확률을 의미하나요?  
**Q3.** 실제 평균이 1,550보다 작은데 귀무가설을 기각하지 못했다면 어떤 오류인가요?

#### 제출 결과

- 가설 설정과 검정 방향 선택 이유
- 표본 수와 표본평균
- 검정 코드
- t통계량과 p-value
- 귀무가설 판단과 분석 결론
- Q1~Q3 답변

In [4]:
# [가설 설정]
# H₀ (귀무가설): Ames 지역 주택의 평균 지상층 생활면적은 1,550 이상이다. (μ ≥ 1,550 또는 μ = 1,550)
# H₁ (대립가설): Ames 지역 주택의 평균 지상층 생활면적은 1,550보다 작다. (μ < 1,550)

# 1. 결측치 제거
gr_liv_area = df["GrLivArea"].dropna()

# 2. 표본 수와 표본평균 확인
n_area = len(gr_liv_area)
mean_area = gr_liv_area.mean()
print(f"[과제 1] 표본 수: {n_area:,}개 | 표본 평균: {mean_area:.2f} sq ft")

# 3. 단일표본 t-검정 수행 (좌측 단측검정: alternative="less")
popmean_target = 1550
alpha = 0.05
t_stat_area, p_val_area = stats.ttest_1samp(
    gr_liv_area, popmean=popmean_target, alternative="less"
)

print(f"t-통계량: {t_stat_area:.4f}")
print(f"p-value : {p_val_area:.4f}")

# 4. 귀무가설 기각 판단
if p_val_area < alpha:
    print(
        f"판단: p-value({p_val_area:.4f}) < {alpha} 이므로 귀무가설(H₀)을 기각합니다."
    )
    print("결론: 부동산 분석가의 주장(평균 면적이 1,550보다 작다)을 지지할 통계적 근거가 있습니다.")
else:
    print(
        f"판단: p-value({p_val_area:.4f}) >= {alpha} 이므로 귀무가설(H₀)을 기각할 수 없습니다."
    )
    print("결론: 분석가의 주장을 지지할 통계적 증거가 불충분합니다.")

[과제 1] 표본 수: 1,460개 | 표본 평균: 1515.46 sq ft
t-통계량: -2.5113
p-value : 0.0061
판단: p-value(0.0061) < 0.05 이므로 귀무가설(H₀)을 기각합니다.
결론: 부동산 분석가의 주장(평균 면적이 1,550보다 작다)을 지지할 통계적 근거가 있습니다.


### 2. 과제 1 해석 질문 답변 (Q1 ~ Q3)

* **Q1. 이 문제에서 적절한 `alternative` 값은 무엇인가요?**  
  **`"less"`** 입니다. 분석가의 주장이 "평균 지상층 생활면적이 1,550보다 **작다($\mu < 1,550$)**"이므로 좌측 꼬리 확률을 검정하는 단측검정을 지정해야 합니다.
* **Q2. p-value는 귀무가설이 참일 확률을 의미하나요?**  
  **아닙니다.** p-value는 "귀무가설이 참이라는 가정하에서($P(\text{Data} \mid H_0)$), 현재 관측된 표본평균과 같거나 더 극단적인 표본 결과가 우연히 관측될 확률"을 의미합니다. 귀무가설 자체가 참일 확률($P(H_0 \mid \text{Data})$)을 나타내는 지표가 아닙니다.
* **Q3. 실제 평균이 1,550보다 작은데 귀무가설을 기각하지 못했다면 어떤 오류인가요?**  
  **제2종 오류 (Type II Error, $\beta$)**입니다. 실제로 대립가설이 참(면적이 1,550보다 작음)인데도, 표본 수 부족이나 높은 데이터 변동성으로 인해 그 차이를 감지하지 못하고 놓쳐버린 오류(False Negative)입니다.

## [실습 마무리] 핵심 5문항 요약

1. **이번 실습에서는 표본평균이 기준값과 다른지 판단하기 위해 어떤 검정을 사용했나요?**  
   모분산을 알지 못하는 상황에서 하나의 연속형 변수 표본평균을 기준 상수값과 비교하기 위해 **단일표본 t-검정 (`scipy.stats.ttest_1samp`)**을 사용했습니다.
2. **귀무가설의 기각 여부는 무엇을 기준으로 판단했나요?**  
   데이터로부터 계산된 **`p-value`**와 분석 시작 전 사전 설정한 허용 오류 기준인 **유의수준 $\alpha$ (0.05)**를 비교하여, $p < \alpha$일 때 귀무가설을 기각하고 $p \ge \alpha$일 때 기각을 보류하는 규칙을 기준으로 삼았습니다.
3. **양측검정과 단측검정은 연구 질문에서 어떤 차이가 있나요?**  
   * **양측검정**: 기준값과의 단순 차이 발생 여부(크든 작든 달라졌는가? $\neq$)에 관심이 있을 때 사용합니다.
   * **단측검정**: 이전보다 증가했는지($>$) 또는 감소했는지($<$)와 같이 명확한 한쪽 방향의 변화에 관심이 있을 때 사용합니다.
4. **귀무가설을 기각하지 못한다는 것은 무엇을 의미하나요?**  
   귀무가설이 100% 옳다고 증명된 것이 아니라, **"현재 확보한 표본 데이터만으로는 귀무가설을 반박할 만한 통계적 증거가 충분하지 않다(기각 보류)"**는 뜻입니다.
5. **제1종 오류와 제2종 오류는 각각 어떤 상황인가요?**  
   * **제1종 오류 ($\alpha$, 헛경보)**: 실제로는 차이가 없는데, 우연의 일치를 보고 "차이가 있다"며 귀무가설을 잘못 기각하는 오류.
   * **제2종 오류 ($\beta$, 불감증)**: 실제로는 명백한 차이가 존재하는데, 증거를 찾아내지 못해 귀무가설을 기각하지 못하고 놓치는 오류.